# Dose rate extraction

In [1]:
# Allows Python to find mosh_src if this notebook is moved
import sys
sys.path.insert(0, "../")

# Add functionality to parse folders
import os

# Import matplotlib and numpy - helpful Python libraries for scientific computing
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.patches as patches
from matplotlib import ticker
from math import pi
from decimal import Decimal

# Import all MOSH plotting functions
from mosh_src.plotting_functions import *

In [2]:
# Dose rate extraction process:
# Iterate over dwell positions (output files) j -->
    # Extract normalization dose rate @ source
    # For each dwell position, iterate over dose points i -->
        # Scale each dose point to rate @ source to generate normed_dose_rate
        # Set D[i][[j] = normed_dose_rate

In [3]:
# Define parameters
dose_points = np.array([(0, 0.25, 0.75), (0, 0.4, 0.8), (0, 0.4, 0.4)]) # cm - note that these are the same across all output files
dwell_pos_transZ = np.array([-0.5, -0.4]) # cm - assuming source is centered in applicator (x, y) = (0, 0)
output_files = np.array(["rate_extract_test_5.csv", "rate_extract_test_4.csv"])

if np.shape(dwell_pos_transZ)[0] != np.shape(output_files)[0]:
    raise ValueError("Number of dwell position Z coordinates must match number of output files")

num_dwell_pos = np.shape(output_files)[0]
num_dose_points = np.shape(dose_points)[0]
source_dose_rate = 10 # Gy/s - same across all output files

# Number of bins and dosing grid half-length - both set in TOPAS parameter files
bins_mosh = 50
dose_grid_half_mosh = 5 # cm

# Computes the lattice length for the simulation space in cm
lattice_mosh = dose_grid_half_mosh*2/bins_mosh

In [4]:
D = np.zeros((num_dose_points, num_dwell_pos))

for j in range(num_dwell_pos):
    # Step 1 - extract normalized dose rate @ source
    source_plane = cm_to_voxel(np.array(dwell_pos_transZ[j]), bins_mosh, lattice_mosh)
    source_energy, __ = extract_norm_energy(output_files[j], source_plane, 2, (0, 0), bins_mosh, lattice_mosh)

    # Step 2 - iterate over dose points, extract energy, and scale to source_norm_energy
    for i in range(num_dose_points):
        dose_point_transZ = dose_points[i][2]
        
        dose_plane = cm_to_voxel(np.array(dose_point_transZ), bins_mosh, lattice_mosh)
        dose_energy, __ = extract_norm_energy(output_files[j], dose_plane, 2, (dose_points[i][0], dose_points[i][1]), bins_mosh, lattice_mosh)

        normed_dose_energy = (dose_energy/source_energy)*source_dose_rate
        D[i][j] = normed_dose_energy

print(D)

FileNotFoundError: rate_extract_test_5.csv not found.

In [ ]:
# Debugging - confirming that D matrix is reasonable given dose points given above
# Step 1 - convert dose points to voxel space
voxel_1 = cm_to_voxel(np.array(dose_points[1]), bins_mosh, lattice_mosh)
source_coord_11 = cm_to_voxel(np.array([0, 0, -0.4]), bins_mosh, lattice_mosh)
print(voxel_1)
print(source_coord_11)
# Choosing dosing point 1, dwell position = -0.5
# (25, 26, 26) = 1.032243290843837e-09
# (25, 25, 23) = source = 5.511913414579794e-08
dose_1 = 4.667749867182103e-11
source = 1.279735784009972e-07

print((dose_1/source)*source_dose_rate) # --> this is consistent with D_22 in matrix above! Function behaving as expected

In [5]:
# Debugging - confirming that test_D_matrix.npy, defined in the end-to-end model,
# contains the correct dose rate matrix
D_load = np.load("test_D_matrix.npy")
print(D_load) # this looks correct in comparison with matrix from the end-to-end model!

[[0.00330148 0.00202058 0.00151083 0.00119017 0.00125357 0.00058261]
 [0.00252969 0.00308175 0.00220237 0.00239871 0.00159268 0.00211737]
 [0.00038234 0.00025499 0.00029085 0.00043986 0.00048639 0.00065843]]
